# Lab 2: Simple PyTorch Image Classifier

This notebook contains a simplified PyTorch version of the same image classification task.

The goal is to keep the code easier to understand while still showing the standard PyTorch workflow:
- prepare the dataset,
- build the CNN,
- train the model,
- and evaluate it.

This version removes extra complexity and focuses on the most important pieces only.

## 1. Import the required libraries

We use PyTorch for training and `torchvision` for image loading.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print("PyTorch is ready")

## 2. Prepare the dataset

The dataset is organized in folders, so `ImageFolder` can automatically assign labels.

In [ ]:
DATASET_PATH = os.path.join('.', 'images_dataSAT')

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

full_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=transform)
print("Classes:", full_dataset.classes)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

## 3. Build the CNN model

This is a simple convolutional neural network for binary classification.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)

## 4. Train the model

The training loop updates the weights using the loss and optimizer.

In [ ]:
for epoch in range(3):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        labels = labels.float().view(-1, 1)
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch {epoch + 1} - loss: {running_loss / len(train_loader):.4f}')

## 5. Evaluate the model

We test the trained model on validation data and compute accuracy.

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        labels = labels.float().view(-1, 1)
        outputs = model(images)
        preds = (outputs >= 0.5).float()
        total += labels.size(0)
        correct += (preds == labels).sum().item()

accuracy = correct / total
print(f'Validation accuracy: {accuracy:.4f}')

## Summary

This simplified PyTorch notebook follows the same logic as the Keras version:
1. prepare the dataset,
2. define a CNN,
3. train it,
4. evaluate performance.

PyTorch is more explicit and gives you greater control over the training loop and model behavior.